# DSML Group 11 — Part 2: Predicting Donor Response

## Abstract
This notebook addresses the binary classification task of predicting whether a donor will respond to an outreach campaign (`TARGET_B = 1` or `0`). We follow a structured workflow: data loading and quality analysis, exploratory data analysis (EDA), preprocessing (missing values, skewness, outliers, encoding, scaling), baseline benchmarking of KNN, Decision Tree, and MLP using 5-fold cross-validation, hyperparameter tuning via RandomizedSearchCV, final model selection, retraining on the full training set, and generation of the Kaggle submission file.

The primary evaluation metric throughout is **F1-Score**. All preprocessing steps that depend on statistics (imputation, scaling, encoding) are fitted exclusively on training folds to prevent data leakage.

---

## Group Member Contribution
| Member | Contribution |
|--------|-------------|
| Member 1 | ... |
| Member 2 | ... |
| Member 3 | ... |
| Member 4 | ... |

---
## Environment Setup — Google Colab / Local Detection

**Why:** Notebooks must mount Google Drive when running in Colab, or use local paths otherwise. Missing this cell costs 1 point in the evaluation.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/DSML_Group11')
    TRAIN_PATH  = BASE / 'donors_descriptive.csv'
    TARGET_PATH = BASE / 'donors_train_target.csv'
    TEST_PATH   = BASE / 'test.csv'
    SAMPLE_PATH = BASE / 'sample_submission.csv'
else:
    # Notebook lives in /luis; data sits in sibling folders at the repo root
    TRAIN_PATH  = Path('..') / 'Descriptive' / 'donors_descriptive.csv'
    TARGET_PATH = Path('..') / 'Predictive'  / 'donors_train_target.csv'
    TEST_PATH   = Path('..') / 'Predictive'  / 'test.csv'
    SAMPLE_PATH = Path('..') / 'Predictive'  / 'sample_submission.csv'

print(f"Running in {'Google Colab' if IN_COLAB else 'local environment'}")
for p in [TRAIN_PATH, TARGET_PATH, TEST_PATH, SAMPLE_PATH]:
    print(f'  {p}  ->  exists: {p.exists()}')

---
## Imports & Global Settings

**Why:** Centralising all imports and the random seed at the top ensures reproducibility across every model and transformation in this notebook. The seed must be set once and reused everywhere.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

from scipy import stats
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import (
    StratifiedKFold, cross_validate, RandomizedSearchCV, PredefinedSplit, train_test_split
)
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report, ConfusionMatrixDisplay, precision_recall_curve,
    roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Libraries loaded. Random seed:", RANDOM_SEED)

In [ ]:
from keni_utils import clean_before_split

print("Imported clean_before_split from Keni utils")

---
# I. Introduction

The Civic Support Alliance (CSA) wants to identify which individuals are most likely to donate in response to an outreach campaign. Instead of contacting everyone in their database, they aim to contact fewer people — but the right people.

**Our goal:** Build a binary classifier that predicts `TARGET_B` (1 = donated, 0 = did not donate).

**Model assessment strategy:** 5-fold stratified cross-validation, evaluated primarily on **F1-Score**. Stratification preserves class proportions across folds given the potential class imbalance.

**Pipeline:** Data loading → EDA → Preprocessing → Baseline benchmarking (KNN, Decision Tree, MLP) → Hyperparameter tuning (RandomizedSearchCV) → Model selection → Final training → Kaggle submission.

---
# II. Data Exploration and Preprocessing

## Step 1 — Data Loading

**What:** Load the descriptive features from the shared URL, the target labels from the local CSV, and the test set.  
**Why:** We need to understand the structure, types, and shape of the data before doing anything else. The target is in a separate file and must be merged via `CONTROL_NUMBER`.  
**What we expect:** ~1,600 training observations, a mix of numerical and categorical columns, and some missing values.

In [ ]:
# Load features, target and test set
df_features = pd.read_csv(TRAIN_PATH,  index_col='CONTROL_NUMBER')
target      = pd.read_csv(TARGET_PATH, index_col='CONTROL_NUMBER')
df_test     = pd.read_csv(TEST_PATH,   index_col='CONTROL_NUMBER')

# Join features with target on CONTROL_NUMBER index
df_train = df_features.join(target)

print('Features shape  :', df_features.shape)
print('Target shape    :', target.shape)
print('Train shape     :', df_train.shape)
print('Test shape      :', df_test.shape)
print('\nTrain columns   :', df_train.columns.tolist())

In [ ]:
# ---- Deterministic cleaning (Keni) — safe before split ----
# Removes impossible values, clips ranges, builds missingness flags.
# NO statistics computed from data here -> no leakage. Stat-based imputation
# happens later inside the CV pipeline (SimpleImputer).
df_train = clean_before_split(df_train)
df_test  = clean_before_split(df_test)

print('Deterministic cleaning applied.')
print(f'Train shape: {df_train.shape}  |  Test shape: {df_test.shape}')
print(f'Remaining NaN in train (to be imputed in pipeline): {int(df_train.isnull().sum().sum())}')

In [ ]:
print("Data types per column:")
print(df_train.dtypes)
print("\nBasic statistics:")
df_train.describe(include='all')

---
## Step 2 — Data Quality Analysis

**What:** Analyse missing values, check for anomalies and inconsistent values, and understand the target class balance.  
**Why:** A model cannot run with missing values or text columns. Knowing the class imbalance upfront tells us whether F1-score is the right primary metric (it is — it handles imbalance better than accuracy).  
**What we expect:** Some missing values in income/demographic fields; possible class imbalance (more non-donors than donors).

In [ ]:
# Missing values analysis
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print(f"Columns with missing values: {len(missing_df)}")
print(missing_df)

In [ ]:
# Target class distribution
target_counts = df_train['TARGET_B'].value_counts()
target_pct    = df_train['TARGET_B'].value_counts(normalize=True) * 100

print("Target distribution:")
print(pd.DataFrame({'Count': target_counts, 'Percentage %': target_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Did not donate (0)', 'Donated (1)'], target_counts.values, color=['#d9534f', '#5cb85c'])
ax.set_title('Target Variable Distribution (TARGET_B)')
ax.set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nClass imbalance ratio: {target_counts[0] / target_counts[1]:.2f}:1 (non-donor:donor)")

In [ ]:
# Identify categorical vs numerical columns (target excluded)
# CONTROL_NUMBER is the index, so it is not among the columns.
cat_cols = df_train.select_dtypes(include='object').columns.tolist()
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()

if 'TARGET_B' in num_cols:
    num_cols.remove('TARGET_B')

print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
print(f'\nNumerical columns ({len(num_cols)}): {num_cols}')

---
## Step 3 — Exploratory Data Analysis (EDA)

**What:** Analyse distributions of numerical features, check for skewness, detect outliers, and explore the relationship between key features and TARGET_B.  
**Why:** EDA must lead to action — every insight should inform a preprocessing decision. Skewed features can hurt distance-based models (KNN) and neural networks. Outliers may distort imputation means and scaling.  
**What we expect:** Some right-skewed gift amount features; outliers in financial variables; clear separability between donors and non-donors on recency/frequency features.

In [ ]:
# Distribution of numerical features
n_cols_plot = 4
n_rows_plot = (len(num_cols) + n_cols_plot - 1) // n_cols_plot

fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(16, n_rows_plot * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df_train[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Numerical Features', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Skewness check for all numerical features
skewness = df_train[num_cols].skew().sort_values(ascending=False)
skew_df = pd.DataFrame({'Skewness': skewness.round(3)})
skew_df['Level'] = pd.cut(
    skew_df['Skewness'].abs(),
    bins=[0, 0.5, 1.0, np.inf],
    labels=['Low (<0.5)', 'Moderate (0.5-1)', 'High (>1)']
)
print("Skewness per numerical feature:")
print(skew_df)

highly_skewed = skew_df[skew_df['Skewness'].abs() > 1].index.tolist()
print(f"\nHighly skewed features (|skew| > 1): {highly_skewed}")

In [ ]:
# Outlier detection using IQR for each numerical feature
outlier_summary = []

for col in num_cols:
    series = df_train[col].dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outliers = ((series < lower) | (series > upper)).sum()
    pct_outliers = n_outliers / len(series) * 100
    outlier_summary.append({
        'Feature': col, 'Q1': round(Q1, 2), 'Q3': round(Q3, 2),
        'IQR': round(IQR, 2), 'Lower fence': round(lower, 2),
        'Upper fence': round(upper, 2), 'N outliers': n_outliers,
        'Outlier %': round(pct_outliers, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier %', ascending=False)
print("Outlier summary (IQR method):")
print(outlier_df[outlier_df['N outliers'] > 0].to_string(index=False))

In [ ]:
# Correlation of numerical features with TARGET_B
correlations = df_train[num_cols + ['TARGET_B']].corr()['TARGET_B'].drop('TARGET_B').sort_values()

fig, ax = plt.subplots(figsize=(8, max(5, len(correlations) * 0.3)))
colors = ['#d9534f' if v < 0 else '#5cb85c' for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation of Numerical Features with TARGET_B')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

print("\nTop 5 positive correlations with TARGET_B:")
print(correlations.tail(5))
print("\nTop 5 negative correlations with TARGET_B:")
print(correlations.head(5))

In [ ]:
# Categorical feature distribution vs TARGET_B
if cat_cols:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 4))
    if len(cat_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        ct = df_train.groupby([col, 'TARGET_B']).size().unstack(fill_value=0)
        ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
        ct_pct.plot(kind='bar', ax=ax, color=['#d9534f', '#5cb85c'], edgecolor='white')
        ax.set_title(f'{col} vs TARGET_B')
        ax.set_ylabel('% within category')
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.tick_params(axis='x', rotation=45)
        ax.legend(['Did not donate', 'Donated'], fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No categorical columns found.")

---
## Step 4 — Data Preparation & Feature Engineering

**What:** Build preprocessing pipelines and engineer new features that capture donor behaviour more directly.  
**Why:** This is the main differentiator between groups. Two improvements beyond basic cleaning:

### Feature Engineering (simple, explainable ratios)
| Feature | Formula | Rationale |
|---------|---------|-----------|
| `RECENCY_X_FREQ` | `FREQUENCY_STATUS / (MONTHS_SINCE_LAST_GIFT + 1)` | High frequency + recent activity = active donor |
| `RESPONSE_RATE` | `RECENT_RESPONSE_COUNT / (NUMBER_PROM_12 + 1)` | Response rate to recent promotions |
| `CARD_RESP_RATE` | `RECENT_CARD_RESPONSE_COUNT / (CARD_PROM_12 + 1)` | Card-specific response rate |
| `WEALTH_KNOWN` | `1 if WEALTH_RATING is not null else 0` | 46% missing — missingness itself is a signal |

### Preprocessing decisions
- **Missing values:** Median for numerical; most-frequent for categorical — fitted on training data only
- **Skewness:** Log1p for features with |skewness| > 1; clip negatives to 0 before log
- **Outliers:** Winsorization (1st/99th percentile) — on training data only, then applied to val/test
- **Encoding:** `OneHotEncoder(drop='first')` — avoids multicollinearity, consistent across splits
- **Scaling:** `MinMaxScaler` — fitted on training data only
- **class_weight='balanced'** for Decision Tree — tells the model the class imbalance (25% positive)
- **Threshold = 0.25** for all models — matches the true class ratio; default 0.5 assumes equal classes

**All steps inside `ColumnTransformer` + `Pipeline` — applied correctly in every CV fold.**

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class Log1pTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to right-skewed features (clips negatives to 0 first)."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return np.log1p(np.clip(X, 0, None))

class WinsorizationTransformer(BaseEstimator, TransformerMixin):
    """Cap at 1st/99th percentile. Percentiles fitted on training data only."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
    def fit(self, X, y=None):
        self.lower_ = np.nanpercentile(X, self.lower * 100, axis=0)
        self.upper_ = np.nanpercentile(X, self.upper * 100, axis=0)
        return self
    def transform(self, X):
        return np.clip(X, self.lower_, self.upper_)

print('Custom transformers defined.')

# ---- Feature Engineering ----
# We engineer ratio features and then keep only those whose |corr| with TARGET_B
# is >= 0.08 (the same threshold used for the original features). Two survive:
#   RECENCY_X_FREQ  (|corr| 0.14 -> the single strongest feature)
#   CARD_RESP_RATE  (|corr| 0.08)
# Others tried (RESPONSE_RATE, LIFETIME_VALUE_TIER, IS_RECENT_DONOR, WEALTH_KNOWN)
# fell below 0.08 and were dropped to cut noise.
def engineer_features(d):
    d = d.copy()
    # Recency x frequency: active donors give often AND recently (strongest feature)
    d['RECENCY_X_FREQ'] = d['FREQUENCY_STATUS_97NK'] / (d['MONTHS_SINCE_LAST_GIFT'] + 1)
    # Card-specific response rate
    d['CARD_RESP_RATE'] = d['RECENT_CARD_RESPONSE_COUNT'] / (d['CARD_PROM_12'] + 1)
    return d

df_train = engineer_features(df_train)
df_test  = engineer_features(df_test)

new_feats = ['RECENCY_X_FREQ', 'CARD_RESP_RATE']
print('Feature engineering applied to train and test.')
print('New features kept (|corr| >= 0.08):', new_feats)

In [ ]:
# ---- Feature Selection ----
# Keep only numerical features whose |corr| with TARGET_B is >= 0.08.
# Everything weaker is dropped as noise (it did not change CV F1 when removed).
# Ordered by |corr| (strongest first):
selected_num_cols = [
    'RECENCY_X_FREQ',            # 0.144  (engineered)
    'FREQUENCY_STATUS_97NK',     # 0.123
    'RECENT_RESPONSE_COUNT',     # 0.121
    'RECENT_CARD_RESPONSE_COUNT',# 0.121
    'RECENT_RESPONSE_PROP',      # 0.119
    'LIFETIME_GIFT_COUNT',       # 0.100
    'PEP_STAR',                  # 0.100
    'FILE_CARD_GIFT',            # 0.099
    'RECENT_CARD_RESPONSE_PROP', # 0.098
    'MONTHS_SINCE_LAST_GIFT',    # 0.089
    'CARD_RESP_RATE',            # 0.083  (engineered)
]

# Categorical features kept for one-hot encoding
selected_cat_cols = ['DONOR_GENDER', 'RECENCY_STATUS_96NK', 'URBANICITY']

feature_cols = selected_num_cols + selected_cat_cols
print(f'Total features: {len(feature_cols)} '
      f'({len(selected_num_cols)} numerical, {len(selected_cat_cols)} categorical)')
print(f'\nNumerical : {selected_num_cols}')
print(f'\nCategorical: {selected_cat_cols}')

In [ ]:
# ---- Preprocessing Pipeline — column by column ----
# One transformer per column so any single column is trivial to adjust.
# All stat-based steps (impute/winsorize/scale) are fitted inside each CV fold
# via the ColumnTransformer -> no leakage.
#   numerical : median impute -> [log1p if skewed] -> winsorize -> MinMax scale
#   categorical: most-frequent impute -> one-hot (drop first)

def num_pipe(log1p=False):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if log1p:
        steps.append(('log1p', Log1pTransformer()))
    steps += [('winsorize', WinsorizationTransformer(lower=0.01, upper=0.99)),
              ('scaler', MinMaxScaler())]
    return Pipeline(steps)

def cat_pipe():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
    ])

# --- Numerical columns: log1p=True for highly skewed (|skew|>1) features ---
numerical_transformers = [
    ('RECENCY_X_FREQ',             num_pipe(log1p=True)),
    ('FREQUENCY_STATUS_97NK',      num_pipe(log1p=False)),
    ('RECENT_RESPONSE_COUNT',      num_pipe(log1p=True)),
    ('RECENT_CARD_RESPONSE_COUNT', num_pipe(log1p=True)),
    ('RECENT_RESPONSE_PROP',       num_pipe(log1p=False)),
    ('LIFETIME_GIFT_COUNT',        num_pipe(log1p=True)),
    ('PEP_STAR',                   num_pipe(log1p=False)),
    ('FILE_CARD_GIFT',             num_pipe(log1p=False)),
    ('RECENT_CARD_RESPONSE_PROP',  num_pipe(log1p=False)),
    ('MONTHS_SINCE_LAST_GIFT',     num_pipe(log1p=False)),
    ('CARD_RESP_RATE',             num_pipe(log1p=True)),
]

# --- Categorical columns ---
categorical_transformers = [(col, cat_pipe()) for col in selected_cat_cols]

# --- Assemble ColumnTransformer (one transformer per column) ---
transformers = [
    (col, pipe, [col])
    for col, pipe in numerical_transformers + categorical_transformers
]
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

print('Preprocessing pipeline assembled (one transformer per column).')
print(f'{len(numerical_transformers)} numerical + {len(categorical_transformers)} categorical '
      f'= {len(transformers)} transformers')

In [ ]:
X = df_train[feature_cols].copy()
y = df_train['TARGET_B'].copy()

# Initial reference threshold = class ratio. The final threshold is tuned later
# on out-of-fold probabilities (Step 8) rather than assumed.
THRESHOLD = 0.25

print(f'X shape          : {X.shape}')
print(f'y shape          : {y.shape}')
print(f'Class ratio      : {y.mean():.3f} positive ({int(y.sum())} donors / {len(y)} total)')
print(f'Reference thresh : {THRESHOLD}  (class ratio; final threshold tuned in Step 8)')

---
# III. Binary Classification & Optimization

## Step 5 — Baseline Benchmarking

**What:** Evaluate KNN, Decision Tree, and MLP with default/light parameters and 5-fold stratified CV.  
**Why:** Benchmark all algorithms before tuning to understand which direction to invest in.  

**Key changes vs naive baseline:**
- Decision Tree uses `class_weight='balanced'` — without it, the tree almost never predicts class 1 (F1 ≈ 0.04 at t=0.5)
- All models evaluated at `THRESHOLD=0.25` (matching class ratio) rather than 0.5
- MLP uses `activation='tanh'` — validated to outperform `relu` on this dataset

In [ ]:
from sklearn.metrics import make_scorer

# F1 evaluated at the reference THRESHOLD (not the default 0.5), so CV scores
# reflect how we actually predict on an imbalanced target.
def f1_at_threshold(y_true, y_proba, threshold=THRESHOLD):
    return f1_score(y_true, (y_proba >= threshold).astype(int))

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Three algorithms covered in class. class_weight='balanced' for the tree, since
# the target is ~25% positive (KNN and MLP do not support class_weight).
baseline_models = {
    'KNN':            KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Decision Tree':  DecisionTreeClassifier(max_depth=5, class_weight='balanced',
                                             random_state=RANDOM_SEED),
    'MLP':            MLPClassifier(hidden_layer_sizes=(100,), activation='tanh',
                                    max_iter=500, early_stopping=True, random_state=RANDOM_SEED),
}

baseline_results = {}
for name, model in baseline_models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    fold_f1_val, fold_f1_train = [], []
    for tr_idx, va_idx in cv_strategy.split(X, y):
        pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        proba_val   = pipe.predict_proba(X.iloc[va_idx])[:, 1]
        proba_train = pipe.predict_proba(X.iloc[tr_idx])[:, 1]
        fold_f1_val.append(f1_at_threshold(y.iloc[va_idx], proba_val))
        fold_f1_train.append(f1_at_threshold(y.iloc[tr_idx], proba_train))
    baseline_results[name] = {
        'Train F1 (mean)': np.mean(fold_f1_train),
        'Val F1 (mean)':   np.mean(fold_f1_val),
        'Val F1 (std)':    np.std(fold_f1_val),
        'Overfit gap':     np.mean(fold_f1_train) - np.mean(fold_f1_val),
    }
    print(f'{name:15s} | Val F1: {np.mean(fold_f1_val):.4f} +/- {np.std(fold_f1_val):.4f} '
          f'| Train F1: {np.mean(fold_f1_train):.4f}')

results_df = pd.DataFrame(baseline_results).T.round(4)
print(f'\nBaseline benchmarking (threshold = {THRESHOLD}):')
results_df

In [ ]:
# Visualise baseline results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

model_names = list(baseline_results.keys())
val_f1   = [baseline_results[m]['Val F1 (mean)'] for m in model_names]
train_f1 = [baseline_results[m]['Train F1 (mean)'] for m in model_names]
val_std  = [baseline_results[m]['Val F1 (std)'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35
axes[0].bar(x - width/2, train_f1, width, label='Train F1', color='steelblue')
axes[0].bar(x + width/2, val_f1,   width, label='Val F1',   color='coral',
            yerr=val_std, capsize=4)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names)
axes[0].set_ylabel('F1-Score'); axes[0].set_title('Baseline: Train vs Validation F1')
axes[0].legend(); axes[0].set_ylim(0, 1)

gaps = [baseline_results[m]['Overfit gap'] for m in model_names]
axes[1].bar(model_names, gaps, color=['#5cb85c' if g < 0.1 else '#d9534f' for g in gaps])
axes[1].set_title('Overfit Gap (Train F1 - Val F1)\nLower is better')
axes[1].set_ylabel('Gap')
axes[1].axhline(0.1, color='gray', linestyle='--', linewidth=0.8, label='0.1 threshold')
axes[1].legend()
plt.tight_layout(); plt.show()

---
## Step 6 — Hyperparameter Tuning

**What:** Use `RandomizedSearchCV` to tune each model over a defined parameter grid.  
**Why:** Default parameters are almost never optimal. We must tune all models before comparing them.  

**Speed strategy — why the previous version was slow (12 min):**
- `n_iter=50` × `5-fold CV` × `3 models` × full pipeline = ~750 pipeline fits
- Fix: use **`PredefinedSplit`** (single validation fold) inside RandomizedSearchCV → ~5x faster
- Reduce `n_iter=25` — still covers a wide enough space
- `MLPClassifier` with `early_stopping=True` stops training when validation loss stops improving
- After finding best params, one final **5-fold `cross_validate`** gives a reliable estimate

**Why this is still valid:**
- We are using PredefinedSplit only to speed up the search, not as the final evaluation
- The final evaluation uses full 5-fold CV with the best params found

In [ ]:
from scipy.stats import randint
from sklearn.metrics import make_scorer

# Tune with full 5-fold CV (reliable). Scoring = F1 at the reference THRESHOLD.
def f1_thresh(y_true, y_proba):
    return f1_score(y_true, (y_proba >= THRESHOLD).astype(int))
thresh_scorer = make_scorer(f1_thresh, response_method='predict_proba')

# Pipeline params use the 'model__' prefix
param_grids = {
    'KNN': {
        'model__n_neighbors': randint(3, 30),
        'model__weights':     ['uniform', 'distance'],
        'model__metric':      ['euclidean', 'manhattan'],
    },
    'Decision Tree': {
        'model__max_depth':         [3, 4, 5, 6, 7, 8, 10],
        'model__min_samples_split': randint(2, 40),
        'model__min_samples_leaf':  randint(1, 20),
        'model__criterion':         ['gini', 'entropy'],
        'model__class_weight':      ['balanced'],
    },
    'MLP': {
        'model__hidden_layer_sizes': [(50,), (100,), (150,), (100, 50)],
        'model__activation':         ['tanh', 'relu'],
        'model__alpha':              [1e-4, 1e-3, 1e-2],
        'model__learning_rate_init': [5e-4, 1e-3, 5e-3],
    },
}

print('Parameter grids defined for', list(param_grids.keys()))
print(f'Scoring: F1 at threshold = {THRESHOLD}')

In [ ]:
import time

tuned_best_params = {}
all_tuning_results = []

model_constructors = {
    'KNN':           KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_SEED),
    'MLP':           MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_SEED),
}

for name, model in model_constructors.items():
    t0 = time.time()
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    search = RandomizedSearchCV(
        pipe, param_distributions=param_grids[name],
        n_iter=20, scoring=thresh_scorer, cv=cv_strategy,
        refit=True, n_jobs=-1, random_state=RANDOM_SEED, return_train_score=True,
    )
    search.fit(X, y)
    elapsed = time.time() - t0
    tuned_best_params[name] = search.best_params_
    best_idx      = search.best_index_
    best_train_f1 = search.cv_results_['mean_train_score'][best_idx]
    best_val_f1   = search.best_score_
    print(f'\n{"="*55}')
    print(f'{name}  [{elapsed:.0f}s]')
    print(f'  Best Val F1   (t={THRESHOLD}) : {best_val_f1:.4f}')
    print(f'  Best Train F1               : {best_train_f1:.4f}')
    print(f'  Overfit gap                 : {best_train_f1 - best_val_f1:.4f}')
    print(f'  Best params                 : {search.best_params_}')
    for i in range(len(search.cv_results_['mean_test_score'])):
        all_tuning_results.append({
            'Model': name,
            'Val F1': search.cv_results_['mean_test_score'][i],
            'Train F1': search.cv_results_['mean_train_score'][i],
            'Overfit gap': search.cv_results_['mean_train_score'][i] - search.cv_results_['mean_test_score'][i],
        })

print('\nTuning complete.')

In [ ]:
# Gold standard: top-10 configurations across all models
tuning_df = pd.DataFrame(all_tuning_results)
top10 = tuning_df.nlargest(10, 'Val F1')[['Model', 'Val F1', 'Train F1', 'Overfit gap']].reset_index(drop=True)
top10['Val F1']      = top10['Val F1'].round(4)
top10['Train F1']    = top10['Train F1'].round(4)
top10['Overfit gap'] = top10['Overfit gap'].round(4)

print("Top 10 configurations across all models:")
print(top10.to_string())

In [ ]:
# ---- Final 5-fold CV with the best params per model ----
# Threshold-aware F1 applied at prediction time. This is the reliable estimate
# used for model selection.
final_cv_results = {}
tuned_pipelines  = {}

for name, best_params in tuned_best_params.items():
    stripped = {k.replace('model__', ''): v for k, v in best_params.items()}
    if name == 'KNN':
        model = KNeighborsClassifier(**stripped)
    elif name == 'Decision Tree':
        model = DecisionTreeClassifier(**stripped, random_state=RANDOM_SEED)
    elif name == 'MLP':
        model = MLPClassifier(**stripped, max_iter=2000, random_state=RANDOM_SEED)

    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    fold_f1_val, fold_f1_train = [], []
    for tr_idx, va_idx in cv_strategy.split(X, y):
        pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        proba_val   = pipe.predict_proba(X.iloc[va_idx])[:, 1]
        proba_train = pipe.predict_proba(X.iloc[tr_idx])[:, 1]
        fold_f1_val.append(f1_at_threshold(y.iloc[va_idx], proba_val))
        fold_f1_train.append(f1_at_threshold(y.iloc[tr_idx], proba_train))
    final_cv_results[name] = {
        'Val F1 (mean)':   np.mean(fold_f1_val),
        'Val F1 (std)':    np.std(fold_f1_val),
        'Train F1 (mean)': np.mean(fold_f1_train),
        'Overfit gap':     np.mean(fold_f1_train) - np.mean(fold_f1_val),
    }
    tuned_pipelines[name] = pipe
    print(f'{name:15s} | Val F1: {np.mean(fold_f1_val):.4f} +/- {np.std(fold_f1_val):.4f} '
          f'| Train F1: {np.mean(fold_f1_train):.4f}')

final_cv_df = pd.DataFrame(final_cv_results).T.round(4)
print(f'\nFinal 5-fold CV results (threshold = {THRESHOLD}):')
final_cv_df

---
## Step 7 — Model Selection

**What:** Choose the final model based on validation F1-score and generalisation (overfit gap).  
**Why:** A model with a slightly lower F1 but a smaller train/validation gap is preferred — it will generalise better to unseen data (the private Kaggle test set that determines our grade).  
**Decision criterion:**
1. Highest mean validation F1-score
2. If two models are close (<0.01 difference), prefer the one with the smaller overfit gap
3. If still tied, prefer the simpler model (fewer parameters, easier to explain at the defence)

In [ ]:
# Select best model: highest Val F1, tie-break on overfit gap
best_row        = final_cv_df.sort_values(['Val F1 (mean)', 'Overfit gap'], ascending=[False, True]).iloc[0]
best_model_name = best_row.name
final_pipeline  = tuned_pipelines[best_model_name]

print(f"Selected model  : {best_model_name}")
print(f"Val F1 (mean)   : {best_row['Val F1 (mean)']:.4f} ± {best_row['Val F1 (std)']:.4f}")
print(f"Train F1 (mean) : {best_row['Train F1 (mean)']:.4f}")
print(f"Overfit gap     : {best_row['Overfit gap']:.4f}")
print(f"\nBest params     : {tuned_best_params[best_model_name]}")

# Visualise final model comparison
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(final_cv_df))
width = 0.35
ax.bar(x - width/2, final_cv_df['Train F1 (mean)'], width, label='Train F1', color='steelblue')
ax.bar(x + width/2, final_cv_df['Val F1 (mean)'],   width, label='Val F1',   color='coral',
       yerr=final_cv_df['Val F1 (std)'], capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(final_cv_df.index)
ax.set_ylabel('F1-Score')
ax.set_title('Tuned Models: Final 5-fold CV Results')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
# IV. Deployment

## Step 8 — Final Training & Prediction

**What:** Retrain the selected model on the full training set, optimise the classification threshold, then generate Kaggle predictions.  
**Why:** Using all available training data maximises information for the final model. Optimising the threshold is critical for F1 — the default 0.5 is almost always suboptimal for imbalanced classes. In the professor's lab the optimal threshold was ≈ 0.334, which significantly improved results.  

**Threshold optimisation process:**
1. Fit the pipeline on a train split (80%)
2. Get predicted probabilities on the holdout (20%)
3. Compute F1 for every possible threshold using `precision_recall_curve`
4. Select the threshold that maximises F1
5. Use that threshold to binarise test predictions

In [ ]:
# ---- Step 8a: Tune the decision threshold on out-of-fold probabilities ----
# Using cross_val_predict, every sample is scored by a model that never saw it,
# so the threshold is chosen without leakage. This is more reliable than a single
# 80/20 split.
from sklearn.model_selection import cross_val_predict

oof_proba = cross_val_predict(final_pipeline, X, y, cv=cv_strategy,
                              method='predict_proba')[:, 1]

precision_t, recall_t, thresholds_t = precision_recall_curve(y, oof_proba)
fscore_t = np.where((precision_t + recall_t) > 0,
                    2 * precision_t * recall_t / (precision_t + recall_t), 0)
best_idx = np.argmax(fscore_t)
FINAL_THRESHOLD = float(thresholds_t[best_idx]) if best_idx < len(thresholds_t) else 0.5

print(f'OOF-optimal threshold : {FINAL_THRESHOLD:.4f}  (F1 = {fscore_t[best_idx]:.4f})')
print(f'Reference threshold   : {THRESHOLD:.4f}     (F1 = {f1_at_threshold(y, oof_proba):.4f})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(recall_t, precision_t, linewidth=1, color='steelblue', label='PR Curve')
ax.scatter(recall_t[best_idx], precision_t[best_idx], s=100, color='crimson',
           zorder=5, label=f'Optimal t = {FINAL_THRESHOLD:.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (out-of-fold) — Threshold Selection')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ---- Step 8b: Retrain the selected model on all training data ----
final_pipeline.fit(X, y)
print(f'Final model retrained on full training set ({len(X)} observations).')

# ---- Step 8c: Predict on the test set using the tuned threshold ----
# df_test went through the same cleaning + feature engineering as df_train.
X_test = df_test[feature_cols].copy()
print(f'\nTest set shape: {X_test.shape}')
print(f'Missing values in test set (imputed in pipeline): {int(X_test.isnull().sum().sum())}')

test_proba = final_pipeline.predict_proba(X_test)[:, 1]
test_predictions = (test_proba >= FINAL_THRESHOLD).astype(int)

print(f'\nThreshold used              : {FINAL_THRESHOLD:.4f}')
print(f'Prediction distribution     : {pd.Series(test_predictions).value_counts().to_dict()}')
print(f'Donation rate in predictions: {test_predictions.mean():.2%}')
print(f'(Training donation rate      = {y.mean():.2%})')

In [ ]:
# Build submission CSV matching sample_submission.csv format
submission = pd.DataFrame({
    'CONTROL_NUMBER': df_test.index,
    'TARGET_B':       test_predictions,
})

# Validate format against sample_submission
sample = pd.read_csv(SAMPLE_PATH)
print('Sample submission columns:', sample.columns.tolist())
print('Our submission columns    :', submission.columns.tolist())
print(f'\nRows in sample    : {len(sample)}')
print(f'Rows in submission: {len(submission)}')
assert list(submission.columns) == list(sample.columns), 'Column mismatch!'
assert len(submission) == len(sample), 'Row count mismatch!'
print('\nFormat validated.')
submission.head()

In [ ]:
# Save submission file next to the other Predictive outputs
submission_path = TEST_PATH.parent / 'DSML_Group11_submission.csv'
submission.to_csv(submission_path, index=False)
print(f'Submission saved to: {submission_path}')

---
# V. Open-Ended Section

## Feature Importance & Error Analysis

**Objective:** Understand what the model learned and where it fails. This goes beyond performance metrics — it connects the model to the business question: who are the people most likely to donate?

**Tasks:**
1. Feature importance (for tree-based models) or permutation importance (for all models)
2. Analysis of misclassified observations — are false negatives (missed donors) or false positives (wrongly targeted) more costly?
3. Prediction probability distribution — how confident is the model?

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance works for all model types
# Use the full training set (pipeline already fitted)
perm_result = permutation_importance(
    final_pipeline, X, y,
    n_repeats=10,
    scoring='f1',
    random_state=RANDOM_SEED,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'Feature':    feature_cols,
    'Importance': perm_result.importances_mean,
    'Std':        perm_result.importances_std,
}).sort_values('Importance', ascending=False)

print("Top 15 features by permutation importance (F1-score drop when shuffled):")
print(perm_df.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
top15 = perm_df.head(15)
ax.barh(top15['Feature'][::-1], top15['Importance'][::-1],
        xerr=top15['Std'][::-1], capsize=3, color='steelblue')
ax.set_title(f'Permutation Feature Importance ({best_model_name})')
ax.set_xlabel('Mean F1-score decrease when feature is shuffled')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix and error analysis on training set predictions
train_preds = final_pipeline.predict(X)

print(f"Classification report on training set:")
print(classification_report(y, train_preds, target_names=['Did not donate', 'Donated']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y, train_preds,
    display_labels=['Did not donate (0)', 'Donated (1)'],
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — Training Set')
plt.tight_layout()
plt.show()

# Analyse misclassified observations
df_errors = df_train.copy()
df_errors['Predicted'] = train_preds
df_errors['Error_type'] = 'Correct'
df_errors.loc[(df_errors['TARGET_B'] == 1) & (df_errors['Predicted'] == 0), 'Error_type'] = 'False Negative (missed donor)'
df_errors.loc[(df_errors['TARGET_B'] == 0) & (df_errors['Predicted'] == 1), 'Error_type'] = 'False Positive (wrongly targeted)'
print(df_errors['Error_type'].value_counts())

In [ ]:
# Prediction probability distribution (if model supports predict_proba)
if hasattr(final_pipeline.named_steps['model'], 'predict_proba'):
    proba = final_pipeline.predict_proba(X)[:, 1]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(proba[y == 0], bins=30, alpha=0.6, color='#d9534f', label='Did not donate (0)')
    ax.hist(proba[y == 1], bins=30, alpha=0.6, color='#5cb85c', label='Donated (1)')
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='Decision boundary (0.5)')
    ax.set_xlabel('Predicted probability of donation')
    ax.set_ylabel('Count')
    ax.set_title('Predicted Probability Distribution by True Class')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not support predict_proba — skipping probability plot.")

---
# Conclusions

## Summary of Results

| Stage | Best Model | Val F1 | Overfit Gap |
|-------|-----------|--------|------------|
| Baseline (default params) | See Step 5 | — | — |
| After tuning (RandomizedSearchCV) | See Step 6 | — | — |
| **Selected for submission** | **See Step 7** | **—** | **—** |

*(Fill in the actual numbers after running the notebook.)*

## Key Takeaways
- Preprocessing quality — particularly handling skewed features and outliers — was the main differentiator
- Fitting all transformers strictly inside cross-validation folds prevented data leakage, making our validation F1 a reliable estimate of test performance
- RandomizedSearchCV allowed exploring a wide hyperparameter space without combinatorial explosion
- The final model was selected based on both validation F1 and overfit gap, not just raw accuracy
- All models were tuned before comparison — results reflect true algorithm differences, not tuning differences

## Kaggle Strategy
- Submissions are identified as `DSML_Group11`
- Internal validation F1 serves as the primary guide — the public Kaggle score reflects only 29% of the test data
- The private score (71%) is what determines the final grade; our cross-validation estimate should be a reliable proxy